# Finnish ASR Baseline: MFCC + GMM-HMM

Team Tokens — COE025 Natural Language Processing, Project 3.

This experiment implements and trains a monophone GMM-HMM recognizer using Kaldi. GMM acoustic distributions and HMM state sequences are components of one classical ASR system, not two separate baseline results. The system is reported as a classical baseline; its implementation and training by the team are documented as project work.

The model uses 13-dimensional MFCC features with delta features, utterance-level mean normalization, a grapheme-based pronunciation approximation, and a train-only add-one unigram language model. Training uses 40 iterations with a target of 1,000 Gaussians.

Test scoring uses the team's shared `asr_eval.py`. Existing Whisper and Qwen results are separate experiments.


## 1. Python Dependencies


In [2]:
%pip -q install jiwer==4.0.0 soundfile==0.13.1 pyarrow huggingface-hub


## 2. Experiment Files

The embedded archive contains the evaluation scripts, fixed manifests, and experiment protocol. Its SHA-256 checksum is verified before extraction.


In [3]:
import base64,hashlib,io,zipfile
from pathlib import Path
PACKAGE=Path('/content/ece_gmm')
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='8e2460c66ae3711a2cad800b144e86aa1eeeb3a40277cc767d373d3695c87a19'
PACKAGE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    assert all((PACKAGE/m.filename).resolve().is_relative_to(PACKAGE.resolve()) for m in archive.infolist())
    archive.extractall(PACKAGE)
print('Dosyalar açıldı.')


Dosyalar açıldı.


## 3. Reference and Scoring Validation

The preparation checks verify the frozen references and shared WER/CER implementation before audio preparation and training.


In [4]:
import os,sys,subprocess,platform,shutil,datetime
from collections import deque
OUT=Path('/content/Ece_GMM_outputs')
WORK=Path('/content/kaldi_work')
CACHE='/content/asr_audio'
OUT.mkdir(parents=True,exist_ok=True)
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['OPENBLAS_NUM_THREADS']='1'
def command(args):
    logs=OUT/'logs';logs.mkdir(exist_ok=True)
    log_path=logs/(datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.log')
    tail=deque(maxlen=50)
    with log_path.open('w',encoding='utf8') as log:
        with subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,encoding='utf8',errors='replace',bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line)
            status=process.wait()
    if status:
        print('SON HATA SATIRLARI:\n'+''.join(tail))
        raise RuntimeError(f'İşlem durdu. Yukarıdaki son hata satırlarını paylaş. Log: {log_path}')
def run(script,*args):command([sys.executable,'-u',PACKAGE/script,*args])
print('Python:',platform.python_version(),'| boş alan:',round(shutil.disk_usage('/content').free/2**30,1),'GB')
assert shutil.disk_usage('/content').free>15*2**30, 'En az 15 GB boş disk gerekli'
run('check_package.py')
sys.path.insert(0,str(PACKAGE))
import asr_eval as E
assert E.score(['one two','three'],['one','three'])['wer']==33.33
print('HAZIRLIK TAMAM — referanslar ve ortak WER/CER kontrol edildi.')


Python: 3.13.18 | boş alan: 107.6 GB
PAKET KONTROLU TAMAM: 12033 referans, ortak normalizasyon, train-only sozluk, ID eslestirme.
HAZIRLIK TAMAM — referanslar ve ortak WER/CER kontrol edildi.


## 4. Kaldi Build and Compatibility Adjustments

Kaldi revision `e02e35f0254bb033fab73d1df99fc34123e31d56` is compiled with OpenBLAS and CUDA disabled.

The recorded session required the `liblapacke-dev` package and a compatibility adjustment in `cblas-wrappers.h`. Eight direct Fortran LAPACK calls were replaced with the corresponding `LAPACK_*` macros. Diagnostic cells and their outputs are preserved as part of the executed session.

These setup cells retain the session's troubleshooting order. Their displayed order is not a validated fresh-runtime installation sequence; the compatibility adjustment must be incorporated into the build workflow for a clean rerun.


In [5]:
command(["apt-get", "install", "-y", "liblapacke-dev"])

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libblas-dev liblapacke libtmglib-dev libtmglib3
Suggested packages:
  liblapack-doc
The following packages will be REMOVED:
  libatlas-base-dev libatlas3-base
The following NEW packages will be installed:
  libblas-dev liblapacke liblapacke-dev libtmglib-dev libtmglib3
0 upgraded, 5 newly installed, 2 to remove and 0 not upgraded.
Need to get 1,208 kB of archives.
After this operation, 23.7 MB disk space will be freed.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libblas-dev amd64 3.12.0-3build1.1 [170 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libtmglib3 amd64 3.12.0-3build1.1 [141 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 liblapacke amd64 3.12.0-3build1.1 [412 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libtmglib-dev amd64 3.12.0-3build1.1 [140 kB]
Get:5 ht

In [9]:
command(['bash',PACKAGE/'install_kaldi.sh'])
print('KALDI KURULUMU TAMAM')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
cd openfst-1.8.4/ && \
./configure --prefix=`pwd` --enable-static --enable-shared --enable-far --enable-ngram-fsts --enable-lookahead-fsts --with-pic CXX="g++" \
	CXXFLAGS="-g -O2 " \
	LDFLAGS="" LIBS="-ldl"
extras/check_dependencies.sh
extras/check_dependencies.sh: python2.7 is not installed
extras/check_dependencies.sh: python3 present
extras/check_dependencies.sh: Configuring python
extras/check_dependencies.sh: ... If you really want to avoid this, add an empty file /content/kaldi/tools/python/.use_default_python and run this script again.
extras/check_dependencies.sh: ... python3 found, making symlink (python3)
ln: failed to create symbolic link '/content/kaldi/tools/python/python3': No such file or directory
extras/check_dependencies.sh: ... ... python2.7 not found, using python3 as python
ln: 

In [10]:
from pathlib import Path

p = Path("/content/kaldi/src/matrix/cblas-wrappers.h")
text = p.read_text()
backup = p.with_suffix(".h.before-lapack-fix")
if not backup.exists():
    backup.write_text(text)

for name in ["stptri", "dtptri", "sgesvd", "dgesvd",
             "ssptri", "dsptri", "ssptrf", "dsptrf"]:
    old, new = name + "_(", "LAPACK_" + name + "("
    if new not in text:
        assert text.count(old) == 1, f"Beklenmeyen dosya içeriği: {name}"
        text = text.replace(old, new)

p.write_text(text)
print("Uyumluluk düzeltmesi uygulandı.")

Uyumluluk düzeltmesi uygulandı.


In [11]:
from pathlib import Path

log = sorted(Path("/content/Ece_GMM_outputs/logs").glob("*.log"))[-1]
satirlar = log.read_text(errors="replace").splitlines()
hatalar = [i for i, s in enumerate(satirlar)
           if "error:" in s.lower() or "undefined reference" in s.lower()]

for i in hatalar[:5]:
    print("\n".join(satirlar[max(0, i-3):i+5]))
if not hatalar:
    print("\n".join(satirlar[-40:]))

c++  -Wl,-rpath=/content/kaldi/tools/openfst-1.8.4/lib -rdynamic  -Wl,-rpath=/content/kaldi/src/lib  lattice-1best.o   ../rnnlm/libkaldi-rnnlm.so  ../nnet3/libkaldi-nnet3.so  ../cudamatrix/libkaldi-cudamatrix.so  ../lat/libkaldi-lat.so  ../lm/libkaldi-lm.so  ../fstext/libkaldi-fstext.so  ../hmm/libkaldi-hmm.so  ../tree/libkaldi-tree.so  ../util/libkaldi-util.so  ../matrix/libkaldi-matrix.so  ../base/libkaldi-base.so /content/kaldi/tools/openfst-1.8.4/lib/libfst.so -L/content/kaldi_openblas/lib -lopenblas -lgfortran -Wl,-rpath=/content/kaldi_openblas/lib -lm -ldl -lpthread  -o lattice-1best
c++  -Wl,-rpath=/content/kaldi/tools/openfst-1.8.4/lib -rdynamic  -Wl,-rpath=/content/kaldi/src/lib  linear-to-nbest.o   ../rnnlm/libkaldi-rnnlm.so  ../nnet3/libkaldi-nnet3.so  ../cudamatrix/libkaldi-cudamatrix.so  ../lat/libkaldi-lat.so  ../lm/libkaldi-lm.so  ../fstext/libkaldi-fstext.so  ../hmm/libkaldi-hmm.so  ../tree/libkaldi-tree.so  ../util/libkaldi-util.so  ../matrix/libkaldi-matrix.so  ../bas

## 5. Data Preparation

Audio is obtained from a fixed Hugging Face dataset revision. Recording IDs, references, audio format, and durations are checked against the manifests. The pronunciation lexicon and language model are constructed from normalized training text only; development and test text do not expand the training vocabulary.


In [12]:
run('prepare_kaldi.py','--work',WORK,'--cache',CACHE)


{
  "protocol": {
    "github_commit": "0e8992b7d24e52200499ccffb991700a3ab8aca8",
    "dataset": "zehraoz36/team-tokens-finnish-asr-data",
    "dataset_revision": "2dad7a2dd2b4e688e673b4b9c2526f8aac750ec7",
    "kaldi_revision": "e02e35f0254bb033fab73d1df99fc34123e31d56",
    "references": "Frozen manifest text column, never normalized twice",
    "hypotheses": "asr_eval.normalize(raw decoder text), exactly once",
    "decoding": "Fixed before test; train-only grapheme lexicon and add-one unigram LM"
  },
  "normalizer": "repo-asr_eval-0e8992b",
  "model": "MFCC + deltas + grapheme GMM-HMM, monophone, 1000 Gaussians",
  "acoustic_units": "Unicode graphemes, explicitly a pronunciation approximation",
  "language_model": "Train-only add-one unigram, including end-of-sentence probability",
  "vocabulary_source": "train normalized text only; no dev/test vocabulary or LM training",
  "training_manifest_digest": "700d79341e5be971828a5162d14a0d809f940d0a6a6a60f50667a3d1456583a5",
  "cmvn": "

## 6. GMM-HMM Training

The training configuration uses MFCC features with deltas, grapheme-based monophone acoustic units, 40 iterations, and a target of 1,000 Gaussians. The training script also constructs the decoding graph. The model and graph are fixed before test evaluation.


In [13]:
command(['bash',PACKAGE/'train_kaldi.sh',WORK])
print('EGITIM TAMAM')


utils/prepare_lang.sh --position-dependent-phones false data/local/dict <UNK> data/local/lang data/lang
Checking data/local/dict/silence_phones.txt ...
--> reading data/local/dict/silence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/silence_phones.txt is OK

Checking data/local/dict/optional_silence.txt ...
--> reading data/local/dict/optional_silence.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/optional_silence.txt is OK

Checking data/local/dict/nonsilence_phones.txt ...
--> reading data/local/dict/nonsilence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/nonsilence_phones.txt is OK

Checking disjoint: silence_phones.txt, nonsilence_phones.txt
--> disjoint property is OK.

Checking data/local/dict/lexicon.txt
--> reading data/local/dict

## 7. Development Smoke Test

Eight recordings from `dev_cv` check the decoding and scoring pipeline. These sample results are not full development-set or test-set results.


In [14]:
run('decode.py','--work',WORK,'--out',OUT,'--split','dev_cv','--smoke')


{
  "system": "gmm-hmm monophone (trained by us)",
  "role": "baseline",
  "checkpoint": "local/kaldi-grapheme-mono",
  "revision": "6c5143402208e6cdf7d1523fab5541e6c925dccc900bdeef42f48600ea8ca7e3",
  "test_set": "dev_cv",
  "clips": 8,
  "audio_hours": 0.007,
  "compute_sec": 12.4,
  "rtf": 0.4656,
  "rtfx": 2.1,
  "hardware": "CPU x86_64",
  "precision": "Kaldi float32",
  "batch_size": 1,
  "settings": "MFCC13+deltas; per-utterance mean CMVN; grapheme monophone GMM-HMM; train-only unigram; 40 iters/1000 Gaussians; beam13/lattice6/acoustic-scale0.083333; CPU1 decode; RTF=decode-only incl CMVN+deltas+lattice+bestpath, excludes precomputed MFCC: not directly comparable with neural RTF; UNK rendered as unk",
  "date": "2026-11-21",
  "wer": 90.2,
  "cer": 45.62,
  "substitutions": 28,
  "deletions": 17,
  "insertions": 1,
  "ref_words": 51
}


## 8. Results and Experiment Archive

The results archive contains source scripts, scores, per-recording predictions, and logs. Audio and model weights are excluded. The training backup is stored separately.


In [15]:
from google.colab import files
def download_results():
    import zipfile
    path=Path('/content/Ece_GMM_sonuclar_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S')+'.zip')
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for source,prefix in [(OUT,''),(PACKAGE,'scripts/ece_gmm')]:
            for p in sorted(source.rglob('*')):
                if p.is_file() and '__pycache__' not in p.parts and p.suffix in {'.py','.sh','.json','.csv','.md','.log'}:
                    z.write(p,Path(prefix)/p.relative_to(source))
    print('Bilgisayara indirme:',path)
    files.download(str(path))
download_results()


Bilgisayara indirme: /content/Ece_GMM_sonuclar_20261121_050949.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from pathlib import Path

klasor = Path("/content/kaldi_work/exp/mono/log")
for desen in ["update.*.log", "align.*.*.log", "acc.*.*.log"]:
    print("\n---", desen, "---")
    bulunan = []
    for dosya in sorted(klasor.glob(desen)):
        for satir in dosya.read_text(errors="replace").splitlines():
            if "WARNING" in satir or "ERROR" in satir:
                bulunan.append(satir)
        if len(bulunan) >= 3:
            break
    print("\n".join(bulunan[:3]) or "Uyarı bulunmadı.")


--- update.*.log ---
WARNING (gmm-est[5.5.1182~2-e02e3]:MleDiagGmmUpdate():mle-diag-gmm.cc:365) Gaussian has too little data but not removing it because it is the last Gaussian: i = 0, occ = 0, weight = 1
WARNING (gmm-est[5.5.1182~2-e02e3]:MleDiagGmmUpdate():mle-diag-gmm.cc:365) Gaussian has too little data but not removing it because it is the last Gaussian: i = 0, occ = 0, weight = 1
WARNING (gmm-est[5.5.1182~2-e02e3]:MleDiagGmmUpdate():mle-diag-gmm.cc:365) Gaussian has too little data but not removing it because it is the last Gaussian: i = 0, occ = 0, weight = 1

--- align.*.*.log ---
WARNING (gmm-align-compiled[5.5.1182~2-e02e3]:AlignUtteranceWrapper():decoder-wrappers.cc:617) Retrying utterance cv_common_voice_fi_22987462 with beam 40
WARNING (gmm-align-compiled[5.5.1182~2-e02e3]:AlignUtteranceWrapper():decoder-wrappers.cc:617) Retrying utterance cv_common_voice_fi_22987463 with beam 40
WARNING (gmm-align-compiled[5.5.1182~2-e02e3]:AlignUtteranceWrapper():decoder-wrappers.cc:617

## 9. Common Voice Test Set — `test_cv`

The fixed test set contains **1,833 recordings**. The complete set is evaluated with the trained model and the shared scoring policy.


In [17]:
assert (OUT/'dev_smoke/scores'/f'{E._slug("gmm-hmm monophone (trained by us)")}.csv').exists(), 'Önce geliştirme kontrolü'
run('decode.py','--work',WORK,'--out',OUT,'--split','test_cv')


{
  "system": "gmm-hmm monophone (trained by us)",
  "role": "baseline",
  "checkpoint": "local/kaldi-grapheme-mono",
  "revision": "6c5143402208e6cdf7d1523fab5541e6c925dccc900bdeef42f48600ea8ca7e3",
  "test_set": "test_cv",
  "clips": 1833,
  "audio_hours": 2.608,
  "compute_sec": 4038.4,
  "rtf": 0.4301,
  "rtfx": 2.3,
  "hardware": "CPU x86_64",
  "precision": "Kaldi float32",
  "batch_size": 1,
  "settings": "MFCC13+deltas; per-utterance mean CMVN; grapheme monophone GMM-HMM; train-only unigram; 40 iters/1000 Gaussians; beam13/lattice6/acoustic-scale0.083333; CPU1 decode; RTF=decode-only incl CMVN+deltas+lattice+bestpath, excludes precomputed MFCC: not directly comparable with neural RTF; UNK rendered as unk",
  "date": "2026-11-21",
  "wer": 80.69,
  "cer": 42.58,
  "substitutions": 6741,
  "deletions": 2388,
  "insertions": 216,
  "ref_words": 11582
}


In [18]:
download_results()


Bilgisayara indirme: /content/Ece_GMM_sonuclar_20261121_061708.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Parliament Test Set — `test_parl`

The fixed test set contains **1,913 recordings**. The complete set is evaluated with the same trained model, decoding graph, and scoring policy.


In [19]:
assert (OUT/'dev_smoke/scores'/f'{E._slug("gmm-hmm monophone (trained by us)")}.csv').exists(), 'Önce geliştirme kontrolü'
run('decode.py','--work',WORK,'--out',OUT,'--split','test_parl')


{
  "system": "gmm-hmm monophone (trained by us)",
  "role": "baseline",
  "checkpoint": "local/kaldi-grapheme-mono",
  "revision": "6c5143402208e6cdf7d1523fab5541e6c925dccc900bdeef42f48600ea8ca7e3",
  "test_set": "test_parl",
  "clips": 1913,
  "audio_hours": 5.545,
  "compute_sec": 17469.7,
  "rtf": 0.8751,
  "rtfx": 1.1,
  "hardware": "CPU x86_64",
  "precision": "Kaldi float32",
  "batch_size": 1,
  "settings": "MFCC13+deltas; per-utterance mean CMVN; grapheme monophone GMM-HMM; train-only unigram; 40 iters/1000 Gaussians; beam13/lattice6/acoustic-scale0.083333; CPU1 decode; RTF=decode-only incl CMVN+deltas+lattice+bestpath, excludes precomputed MFCC: not directly comparable with neural RTF; UNK rendered as unk",
  "date": "2026-11-21",
  "wer": 87.2,
  "cer": 47.86,
  "substitutions": 20865,
  "deletions": 8471,
  "insertions": 493,
  "ref_words": 34209
}


In [20]:
download_results()


Bilgisayara indirme: /content/Ece_GMM_sonuclar_20261121_110819.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. FLEURS Test Set — `test_fleurs`

The fixed test set contains **916 recordings**. The complete set is evaluated with the same trained model, decoding graph, and scoring policy.


In [21]:
assert (OUT/'dev_smoke/scores'/f'{E._slug("gmm-hmm monophone (trained by us)")}.csv').exists(), 'Önce geliştirme kontrolü'
run('decode.py','--work',WORK,'--out',OUT,'--split','test_fleurs')


{
  "system": "gmm-hmm monophone (trained by us)",
  "role": "baseline",
  "checkpoint": "local/kaldi-grapheme-mono",
  "revision": "6c5143402208e6cdf7d1523fab5541e6c925dccc900bdeef42f48600ea8ca7e3",
  "test_set": "test_fleurs",
  "clips": 916,
  "audio_hours": 3.285,
  "compute_sec": 6980.4,
  "rtf": 0.5902,
  "rtfx": 1.7,
  "hardware": "CPU x86_64",
  "precision": "Kaldi float32",
  "batch_size": 1,
  "settings": "MFCC13+deltas; per-utterance mean CMVN; grapheme monophone GMM-HMM; train-only unigram; 40 iters/1000 Gaussians; beam13/lattice6/acoustic-scale0.083333; CPU1 decode; RTF=decode-only incl CMVN+deltas+lattice+bestpath, excludes precomputed MFCC: not directly comparable with neural RTF; UNK rendered as unk",
  "date": "2026-11-21",
  "wer": 93.11,
  "cer": 53.07,
  "substitutions": 10557,
  "deletions": 2859,
  "insertions": 257,
  "ref_words": 14685
}


In [22]:
download_results()


Bilgisayara indirme: /content/Ece_GMM_sonuclar_20261121_130440.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Training Backup

The separate compressed training archive contains the data preparation files, MFCC features, trained acoustic model, decoding graph, configuration, and recipe. It is distinct from the smaller results archive.

The uploaded archive passed a complete gzip integrity check. The acoustic-model and decoding-graph SHA-256 hashes match the result metadata:

- Model: `6c5143402208e6cdf7d1523fab5541e6c925dccc900bdeef42f48600ea8ca7e3`
- Graph: `af51378d3a71cf01f682c3d3594a7e17f13d14ceaa8118d0bf12a811e855a1ff`


In [23]:
import tarfile
backup=Path('/content/Ece_GMM_egitim_yedegi.tar.gz')
with tarfile.open(backup,'w:gz') as tar:
    for name in ['data','mfcc','exp','conf','recipe.json']:
        if (WORK/name).exists():tar.add(WORK/name,arcname=name)
files.download(str(backup))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Results and Limitations

All 4,662 test recordings have predictions with unique IDs and references matching the fixed manifests. No empty predictions were found. Each test log reports zero failures in lattice best-path extraction. Corpus-level WER and CER were independently recomputed from the exported predictions and agree with the score files.

| Test set | Recordings | WER (%) | CER (%) | Recorded decoding time (min) |
|---|---:|---:|---:|---:|
| Common Voice (`test_cv`) | 1,833 | 80.69 | 42.58 | 67.3 |
| Parliament (`test_parl`) | 1,913 | 87.20 | 47.86 | 291.2 |
| FLEURS (`test_fleurs`) | 916 | 93.11 | 53.07 | 116.3 |

The three test sets are reported separately. References use the manifest's existing `text` column; raw hypotheses are normalized exactly once by the shared evaluator. The decoder's `<UNK>` symbol is rendered as `unk` before normalization so that annotation cleanup does not remove that recognition error.

Recognition accuracy is low. Relevant limitations include the small training set, monophone acoustic modeling, grapheme-based pronunciation approximation, and a closed vocabulary with a simple unigram language model. These factors are limitations, not a demonstrated causal decomposition of the errors.

The train-only vocabulary contains 6,235 words. The recorded out-of-vocabulary rates measure reference word tokens missing from that vocabulary:

| Test set | OOV tokens / reference words | OOV rate (%) |
|---|---:|---:|
| Common Voice | 4,274 / 11,582 | 36.90 |
| Parliament | 14,944 / 34,209 | 43.68 |
| FLEURS | 8,571 / 14,685 | 58.37 |

The recorded runs use a single CPU decoding process, beam 13, lattice beam 6, and acoustic scale 0.083333. RTF includes CMVN, delta computation, lattice generation, and best-path extraction, but excludes the precomputed MFCC extraction. These CPU timings are not directly comparable with neural GPU inference timings.

The high error rates remain valid baseline findings; they are retained without removing difficult recordings or replacing reference transcripts.
